In [48]:
%pip install -q langchain langchain_openai langgraph langsmith

In [49]:
import os
from google.colab import userdata

os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "triage-capstone"
os.environ["LANGSMITH_API_KEY"] = userdata.get("LANGSMITH_API_KEY")

In [50]:
%pip install -q langchain_huggingface sentence-transformers langchain-text-splitters

In [51]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

kb_texts = [
    """# Credential Access Patterns

## Brute Force / Password Guessing
Repeated failed authentication attempts against the same account or service in
a short time window, especially against privileged or default accounts
(admin, root, administrator), often from a single external IP or a small
rotating set of IPs. A high attempt count in under a minute suggests
automated tooling rather than a human mistyping a password. Severity should
scale with attempt volume, account privilege, and whether any attempt
eventually succeeded.

## Successful Login Anomalies
A successful login itself is not inherently suspicious. Context that raises
severity: new device or geography inconsistent with the user's history,
login immediately following a failed-attempt burst, or a login at an unusual
hour for that account's normal pattern.""",

    """# Malware, Persistence, and Exfiltration Patterns

## Command-and-Control (C2) Indicators
Outbound traffic to newly registered domains, or destinations already known
as C2 infrastructure. Living-off-the-land tools (powershell.exe, wscript.exe)
making outbound network connections shortly after spawning from an office
document process (winword.exe, excel.exe) is a strong indicator of a
maldoc-triggered infection chain and should be treated as high severity.

## Persistence Mechanisms
Scheduled tasks or registry run keys created with generic or disguised names
shortly after suspicious process activity. Deletion of volume shadow copies
(vssadmin delete shadows) is strongly associated with ransomware staging and
should always be treated as critical severity.

## Web Application Attacks
Classic injection patterns (SQL injection strings, path traversal) in HTTP
request parameters. A single rejected probe is low-to-medium severity; any
successful injection is high severity."""
]

docs = [Document(page_content=t) for t in kb_texts]
splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=120)
chunks = splitter.split_documents(docs)

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")
vector_store = InMemoryVectorStore(embeddings)
vector_store.add_documents(chunks)

def retrieve_context(query, k=3):
    results = vector_store.similarity_search(query, k=k)
    return "\n\n".join(d.page_content for d in results)

print("Knowledge base ready.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Knowledge base ready.


In [52]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model="nvidia/nemotron-3-nano-30b-a3b:free",
    temperature=0,
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ.get("OPENROUTER_API_KEY"),
)

In [53]:
from typing import Literal, TypedDict
from langchain_core.tools import tool

@tool
def record_classification(
    threat_category: Literal["credential_access", "malware_c2", "web_attack", "phishing", "benign", "unknown"],
    severity: Literal["low", "medium", "high", "critical"],
    rationale: str,
) -> str:
    """Record the log's threat category, severity, and rationale."""
    return "Classification recorded."

class Feedback(TypedDict):
    grade: Literal["accept", "revise"]
    feedback: str

evaluator = model.with_structured_output(Feedback)
model_with_tools = model.bind_tools([record_classification])

In [54]:
from langgraph.func import entrypoint, task
from langgraph.types import interrupt, RetryPolicy
from langgraph.checkpoint.memory import InMemorySaver

@task(retry_policy=RetryPolicy(max_attempts=3, initial_interval=1.0))
def retrieve_context_task(log_text: str) -> str:
    """Search knowledge base for relevant threat patterns."""
    return retrieve_context(log_text)

@task
def classify_log(log_text: str, context: str, feedback: Feedback = None) -> dict:
    """Real tool call: the LLM must call record_classification itself."""
    prompt = f"""
    Classify this security log entry by calling record_classification.
    Log: {log_text}

    Reference threat patterns:
    {context}

    Write the rationale in plain, simple language, not overly technical.
    """
    if feedback:
        prompt += f"\nTake into account this feedback: {feedback}"

    error_note = ""
    for attempt in range(3):
        if error_note:
            prompt += f"\nYour previous attempt failed with: {error_note}. Try again."
        response = model_with_tools.invoke(prompt)
        if response.tool_calls:
            return response.tool_calls[0]["args"]
        error_note = "You must call record_classification, not just reply in text."

    return {"threat_category": "unknown", "severity": "medium", "rationale": f"Classification failed: {error_note}"}

@task
def llm_call_evaluator(log_text: str, classification: dict) -> Feedback:
    """LLM evaluates the classification"""
    prompt = f"Grade this classification for log '{log_text}': {classification}"
    return evaluator.invoke(prompt)

@task
def escalate_to_human(log_text: str, classification: dict):
    """Pause for human review and approval."""
    decision = interrupt({
        "action": "Please review this triage classification",
        "log_text": log_text,
        "threat_category": classification["threat_category"],
        "severity": classification["severity"],
        "rationale": classification["rationale"],
    })
    return decision

In [55]:
checkpointer = InMemorySaver()

@entrypoint(checkpointer=checkpointer)
def triage_workflow(inputs: dict) -> dict:
    """Main workflow routing logic."""
    log_text = inputs["log_text"]
    context = retrieve_context_task(log_text).result()

    feedback = None
    while True:
        classification = classify_log(log_text, context, feedback).result()
        feedback = llm_call_evaluator(log_text, classification).result()
        if feedback["grade"] == "accept":
            break

    needs_review = classification["severity"] in ["high", "critical"]

    if needs_review:
        decision = escalate_to_human(log_text, classification).result()
        return {"status": "escalated_to_human", "classification": classification, "decision": decision}

    return {"status": "auto_resolved", "classification": classification}

In [56]:
import uuid
from langgraph.types import Command

def triage_log_interactive():
    log_text = input("Paste a log line to triage: ")
    thread_id = str(uuid.uuid4())
    config = {"configurable": {"thread_id": thread_id}}

    result = triage_workflow.invoke({"log_text": log_text}, config)

    if "__interrupt__" in result:
        payload = result["__interrupt__"][0].value
        print("\n>>> ESCALATED TO ANALYST <<<")
        print(payload)
        analyst_response = input("\nYou are the analyst. Type your decision: ")
        result = triage_workflow.invoke(Command(resume=analyst_response), config)

    print("\n--- Final result ---")
    print(result)

triage_log_interactive()

Paste a log line to triage: 10000 login tries

>>> ESCALATED TO ANALYST <<<
{'action': 'Please review this triage classification', 'log_text': '10000 login tries', 'threat_category': 'credential_access', 'severity': 'high', 'rationale': "There were 10,000 login attempts, which is a very large number of failed logins in a short period. This pattern matches a brute-force attack where an attacker is trying many passwords to guess one. Because the volume is high, it raises serious concern, even though we don't know if any attempt succeeded."}

You are the analyst. Type your decision: block

--- Final result ---
{'status': 'escalated_to_human', 'classification': {'threat_category': 'credential_access', 'severity': 'high', 'rationale': "There were 10,000 login attempts, which is a very large number of failed logins in a short period. This pattern matches a brute-force attack where an attacker is trying many passwords to guess one. Because the volume is high, it raises serious concern, even t